# Figure: MOLLI, ShMOLLI and SASHA side by side

The cell below is tagged `#| label: fig-side-by-side-output`, so `03-molli-shmolli-sasha.md` embeds its output with `:::{figure} #fig-side-by-side-output`.

In [ ]:
#| label: fig-side-by-side-output
# MOLLI, ShMOLLI and SASHA side by side: what happens during the scan, and the fitted curve.
# The simulation itself is in cardiac_sim.py (shared with the figures in the next section).
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings, logging
# Kaleido (the PNG export for the PDF) can print harmless clean-up warnings on Windows.
warnings.filterwarnings("ignore", message=".*temporary directory.*")
for name in ("choreographer", "kaleido", "logistro"):
    logging.getLogger(name).setLevel(logging.CRITICAL)
import os, sys
sys.path.append(os.path.join(os.getcwd(), "notebooks"))  # find cardiac_sim.py also when run from the book folder
from cardiac_sim import SEQUENCES, simulate, fit

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

RR = 1000.0            # time between two heartbeats [ms] (60 beats per minute)
TISSUES = {"native myocardium (T1 = 1000 ms)": 1000.0, "blood (T1 = 1600 ms)": 1600.0,
           "myocardium after contrast (T1 = 400 ms)": 400.0}
PREP_COLOR = {"MOLLI": "royalblue", "ShMOLLI": "royalblue", "SASHA": "darkorange"}
Y_RANGE = [-1.4, 1.3]  # same y-axis left and right, so each image is at the same height
# One colour per group of images, used in BOTH panels so you can follow each image
GROUP_COLOR = {"inversion 1": "#d62728", "inversion 2": "#2ca02c", "inversion 3": "#9467bd",
               "no preparation": "#7f7f7f", "saturation": "black"}

def image_groups(seq, events):
    """For every image (in time order): which preparation it belongs to."""
    groups, n_inv, n_sat = [], 0, 0
    for t_ev, kind in events:
        if kind == "inv":
            n_inv += 1
        elif kind == "sat":
            n_sat += 1
        else:
            if seq == "SASHA":
                groups.append("saturation" if n_sat > 0 else "no preparation")
            else:
                groups.append(f"inversion {n_inv}")
    return groups

# One row per sequence. Left: the scan over time (shared time axis, so the lengths can be
# compared directly). Right: all images sorted by their time after the preparation pulse.
fig = make_subplots(rows=3, cols=2, column_widths=[0.62, 0.38], shared_xaxes="columns",
                    horizontal_spacing=0.08, vertical_spacing=0.06)

TRACES = 8
shown_in_legend = set()
annotations_per_tissue = []
for i, (tissue, T1) in enumerate(TISSUES.items()):
    notes = []
    for row, seq in enumerate(SEQUENCES, start=1):
        vis = (i == 0)
        first = (row == 1)
        t_c, mz_c, events, samples = simulate(seq, T1, RR)
        t0 = events[0][0] - 300                  # start of the plotted time axis
        # ECG: one R-peak per heartbeat
        ecg_x, ecg_y = [], []
        for b in np.arange(np.ceil(t0 / RR) * RR, t_c[-1], RR):
            ecg_x += [(b - 40 - t0) / 1000, (b - t0) / 1000, (b + 40 - t0) / 1000, None]
            ecg_y += [-1.35, -1.12, -1.35, None]
        fig.add_trace(go.Scatter(x=ecg_x, y=ecg_y, mode="lines", line=dict(color="gray"),
                                 name="heartbeat (ECG)", legendgroup="ecg", showlegend=first,
                                 hoverinfo="skip", visible=vis), row=row, col=1)
        fig.add_trace(go.Scatter(x=(t_c - t0) / 1000, y=mz_c, mode="lines",
                                 line=dict(color="lightgray", width=3), name="Mz",
                                 legendgroup="mz", showlegend=first, hoverinfo="skip",
                                 visible=vis), row=row, col=1)
        px, py = [], []
        for t_ev, kind in events:
            if kind != "img":
                px += [(t_ev - t0) / 1000] * 2 + [None]; py += [-1.1, 1.1, None]
        is_inv = seq != "SASHA"
        fig.add_trace(go.Scatter(x=px, y=py, mode="lines",
                                 line=dict(color=PREP_COLOR[seq], dash="dot"),
                                 name="inversion pulse" if is_inv else "saturation pulse",
                                 legendgroup="inv" if is_inv else "sat",
                                 showlegend=(first or seq == "SASHA"), hoverinfo="skip",
                                 visible=vis), row=row, col=1)
        # Mz just before each inversion: shows whether it had time to recover to 1
        inv_t = [t_ev for t_ev, kind in events if kind == "inv"]
        inv_mz = [float(np.interp(t_ev - 1e-6, t_c, mz_c)) for t_ev in inv_t]
        fig.add_trace(go.Scatter(x=[(v - t0) / 1000 for v in inv_t], y=inv_mz,
                                 mode="markers+text", marker=dict(size=7, color="royalblue", symbol="diamond"),
                                 text=[f"{v:.2f}" for v in inv_mz], textposition="top center",
                                 name="Mz just before the inversion", legendgroup="before",
                                 showlegend=first, hoverinfo="skip", visible=vis), row=row, col=1)

        # the fit, and which images were used
        model, p, T1_fit, n_used = fit(seq, samples, RR)
        groups = image_groups(seq, events)
        x_eff = np.array([s[0] for s in samples])
        mz_img = [s[1] for s in samples]
        colors = [GROUP_COLOR[g] for g in groups]
        symbols = ["circle"] * n_used + ["circle-open"] * (len(samples) - n_used)
        label = "TS" if seq == "SASHA" else "TI"
        hover = []
        for k, (g, x, m) in enumerate(zip(groups, x_eff, mz_img)):
            when = "fully recovered" if np.isinf(x) else f"{label} = {x:.0f} ms"
            unused = "<br>(not used in the fit)" if k >= n_used else ""
            hover.append(f"Image {k + 1}, {g}<br>{when}<br>Mz = {m:.2f}{unused}")

        # left: the images at the moment they were taken
        img_t = [(t_ev - t0) / 1000 for t_ev, kind in events if kind == "img"]
        fig.add_trace(go.Scatter(x=img_t, y=mz_img, mode="markers",
                                 marker=dict(size=6, color=colors, symbol=symbols,
                                             line=dict(width=1.5, color=colors)),
                                 text=hover, hovertemplate="%{text}<extra></extra>",
                                 name="image", showlegend=False, visible=vis), row=row, col=1)
        # right: ideal recovery without any readouts, for comparison (drawn behind the images)
        xf = np.linspace(0, 4500, 300)
        ideal = 1 - np.exp(-xf / T1) if seq == "SASHA" else 1 - 2 * np.exp(-xf / T1)
        fig.add_trace(go.Scatter(x=xf, y=ideal, mode="lines", line=dict(color="lightgray", width=5),
                                 name="ideal recovery (no readouts)", legendgroup="ideal",
                                 showlegend=first, hoverinfo="skip", visible=vis), row=row, col=2)
        # right: the same images, sorted by their time after the preparation pulse
        x_plot = np.where(np.isinf(x_eff), 4400, x_eff)   # SASHA's first image at the right edge
        fig.add_trace(go.Scatter(x=x_plot, y=mz_img, mode="markers",
                                 marker=dict(size=6, color=colors, symbol=symbols,
                                             line=dict(width=1.5, color=colors)),
                                 text=hover, hovertemplate="%{text}<extra></extra>",
                                 name="image", showlegend=False, visible=vis), row=row, col=2)
        xf = np.linspace(0, 4500, 300)
        fig.add_trace(go.Scatter(x=xf, y=model(xf, *p), mode="lines",
                                 line=dict(color="dimgray", dash="dash"), name="fitted curve",
                                 legendgroup="fit", showlegend=first, hoverinfo="skip",
                                 visible=vis), row=row, col=2)
        # the fitted T1 as text in the right panel of this row
        axis = 2 * row                                   # right-hand panels are x2, x4, x6
        err = 100 * (T1_fit - T1) / T1
        err = 0.0 if abs(err) < 0.05 else err           # avoid printing '-0.0%'
        notes.append(dict(xref=f"x{axis} domain", yref=f"y{axis} domain", x=0.98, y=0.04,
                          xanchor="right", yanchor="bottom", showarrow=False,
                          text=f"T1 = {T1_fit:.0f} ms ({err:+.1f}%)",
                          bgcolor="white", font=dict(size=12)))
    annotations_per_tissue.append(notes)

# legend entries for the image colours (dummy traces, always visible)
for g, c in GROUP_COLOR.items():
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=6, color=c),
                             name=f"image after {g}" if g != "no preparation" else "image without preparation",
                             hoverinfo="skip"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                         marker=dict(size=6, color="gray", symbol="circle-open", line=dict(width=1.5)),
                         name="image not used in the fit", hoverinfo="skip"))
n_dummy = len(GROUP_COLOR) + 1

buttons = []
for i, tissue in enumerate(TISSUES):
    n = TRACES * len(SEQUENCES)
    visible = [False] * (n * len(TISSUES)) + [True] * n_dummy
    visible[i * n:(i + 1) * n] = [True] * n
    buttons.append(dict(label=tissue, method="update",
                        args=[{"visible": visible}, {"annotations": annotations_per_tissue[i]}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", x=0, xanchor="left", y=1.06, yanchor="bottom")],
    annotations=annotations_per_tissue[0],
    height=950, template="plotly_white", margin=dict(t=70, b=190),
    # legend at the very bottom of the figure, with space below the x-axis titles
    legend=dict(orientation="h", yref="container", yanchor="bottom", y=0.01, x=0),
)
for row, seq in enumerate(SEQUENCES, start=1):
    fig.add_hline(y=1, line=dict(color="gray", width=1, dash="dash"), row=row, col=1)  # full recovery
    fig.update_yaxes(title_text=f"<b>{seq}</b><br>Mz / M0", range=Y_RANGE, row=row, col=1)
    fig.update_yaxes(range=Y_RANGE, row=row, col=2)
fig.update_xaxes(title_text="Time since the start of the scan [s]", row=3, col=1)
fig.update_xaxes(title_text="TI or TS [ms]", range=[0, 4600], row=3, col=2)

# Only the images show their values on hover; no zooming, legend not clickable.
fig.update_layout(hovermode="closest", dragmode=False,
                  legend=dict(itemclick=False, itemdoubleclick=False))
fig.update_xaxes(fixedrange=True)
fig.update_yaxes(fixedrange=True)
fig
